# Ninapro DB5: Controlled Attribution Study (Mamba vs CfC)
This notebook replicates the DB1 experimental protocol on Ninapro DB5 on a remote GPU (e.g., T4 on Google Colab or Kaggle).

### Protocol Summary:
- 5 Models in Execution Order: `cfc-only`, `mamba-only`, `mamba-cfc`, `mamba-width` (W=88), `mamba-3l`
- Window: 300 ms (60 samples @ 200 Hz), Stride: 100 ms (20 samples)
- Subject split: Train [9, 2, 6, 1, 8, 3], Val [10, 5], Test [4, 7]
- Normalization: Frozen channel-wise Z-score from training set (Zero leakage)
- Training: CrossEntropyLoss, Adam lr=1e-3, batch size 256, WeightedRandomSampler, grad clip 1.0, patience 5 on val Macro-F1

## 1. Setup & Repository Ingestion

In [ ]:
# Option A: Clone from public GitHub repository
!git clone https://github.com/rohilgirish/Mamba-CfC-for-Biosignal-Classification.git
%cd Mamba-CfC-for-Biosignal-Classification

# Option A (Private Repo): If repo is private, store GITHUB_TOKEN in Colab Secrets
# from google.colab import userdata
# token = userdata.get("GITHUB_TOKEN")
# !git clone https://{token}@github.com/rohilgirish/Mamba-CfC-for-Biosignal-Classification.git
# %cd Mamba-CfC-for-Biosignal-Classification

# Option B: If using code_db5.zip uploaded to Google Drive (Fallback)
# from google.colab import drive
# drive.mount("/content/drive")
# !unzip -q /content/drive/MyDrive/Ninapro_DB5/code_db5.zip -d /content/ninapro_db5
# %cd /content/ninapro_db5

# Install exact dependencies
!pip install --quiet h5py==3.16.0 ncps==1.0.1 scikit-learn pyyaml

In [ ]:
import torch
import numpy as np
import h5py
import ncps

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"h5py: {h5py.__version__}")
print(f"ncps: {ncps.__version__}")

## 2. Mount Google Drive and Copy Processed Data

In [ ]:
from google.colab import drive
import os

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

DRIVE_DATA = "/content/drive/MyDrive/Ninapro_DB5/data"
LOCAL_DATA = "data/processed_db5"
os.makedirs(LOCAL_DATA, exist_ok=True)

# Copy files to local NVMe SSD for fast data loading
for f in ["train.h5", "val.h5", "test.h5", "norm_stats_db5.json"]:
    src = os.path.join(DRIVE_DATA, f)
    dst = os.path.join(LOCAL_DATA, f)
    if os.path.exists(src) and not os.path.exists(dst):
        !cp "$src" "$dst"
        print(f"Copied {f} to local workspace.")
    elif os.path.exists(dst):
        print(f"{f} ready locally.")
    else:
        print(f"[!] Warning: {src} not found on Google Drive!")

## 3. GPU Latency Profiling & Time Estimation (Seconds per Batch)

In [ ]:
import time
import torch
from src.models.architecture_db5 import get_model_db5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dummy_x = torch.randn(256, 60, 16, device=device)
models = ["cfc-only", "mamba-only", "mamba-cfc", "mamba-width", "mamba-3l"]

print(f"Benchmarking batch latency (BS=256, Device={device})...")
print("-" * 65)
for mname in models:
    m = get_model_db5(mname, pool_mode="last", device=device)
    m.train()
    optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss()
    dummy_y = torch.randint(0, 53, (256,), device=device)
    
    # Warmup
    for _ in range(3):
        optimizer.zero_grad()
        out = m(dummy_x)
        loss = criterion(out, dummy_y)
        loss.backward()
        optimizer.step()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    # Timing 10 iterations
    t0 = time.time()
    for _ in range(10):
        optimizer.zero_grad()
        out = m(dummy_x)
        loss = criterion(out, dummy_y)
        loss.backward()
        optimizer.step()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = (time.time() - t0) / 10.0
    
    # 186,942 train windows / 256 = ~731 batches per epoch
    epoch_min = (dt * 731) / 60.0
    print(f"{mname:<12} | {dt*1000:6.1f} ms/batch | Est. {epoch_min:4.1f} min/epoch | Est. 10 epochs: {epoch_min*10:5.1f} min")
print("-" * 65)

## 4. Run Full Training (Order: cfc-only, mamba-only, mamba-cfc, mamba-width, mamba-3l)

In [ ]:
%%bash
mkdir -p /content/drive/MyDrive/Ninapro_DB5/checkpoints
mkdir -p /content/drive/MyDrive/Ninapro_DB5/reports

MODELS=("cfc-only" "mamba-only" "mamba-cfc" "mamba-width" "mamba-3l")

for model in "${MODELS[@]}"; do
    echo "=========================================================="
    echo "Starting training for: $model"
    echo "=========================================================="
    python scripts/train_db5.py \
        --model "$model" \
        --data-dir data/processed_db5 \
        --out-dir . \
        --max-epochs 50 \
        --batch-size 256 \
        --lr 1e-3 \
        --patience 5 \
        --seed 42 \
        --device cuda \
        --resume
        
    # Backup checkpoints and history after each model
    cp -r checkpoints/* /content/drive/MyDrive/Ninapro_DB5/checkpoints/
    cp -r reports/* /content/drive/MyDrive/Ninapro_DB5/reports/
    echo "Backed up artifacts for $model to Google Drive."
done

## 5. Evaluate Best Checkpoints on Held-Out Test Set (65,237 windows)

In [ ]:
!python scripts/evaluate_db5.py \
    --models cfc-only mamba-only mamba-cfc mamba-width mamba-3l \
    --data-dir data/processed_db5 \
    --ckpt-dir checkpoints \
    --out-dir reports \
    --seed 42 \
    --device cuda

## 6. Display Summary Table & Attribution Analysis

In [ ]:
import json
import pandas as pd

with open("reports/final_results_db5.json", "r") as f:
    results = json.load(f)

rows = []
for model, seeds in results.items():
    for seed, m in seeds.items():
        rows.append({
            "Model": model,
            "Seed": seed,
            "Parameters": f"{m['parameters']:,}",
            "Macro-F1": f"{m['macro_f1']:.4f}",
            "Balanced Accuracy": f"{m['balanced_accuracy']:.4f}"
        })

df = pd.DataFrame(rows)
print(df.to_markdown(index=False))